# Parsli — backend playground

Same five-part flow using backend imports.

| Part | What it does | Backend import |
|---|---|---|
| 1 | Gmail auth + email download | `parsli.gmail` |
| 2 | Deterministic preprocessing | `parsli.processing.cleaner`, `.rule_engine` |
| 3 | Model classification (LM Studio or llama-cpp) | `parsli.model` |
| 4 | Persistence + shipment resolution | `parsli.db`, `parsli.services` |
| 5 | Dashboard projection — UI-ready summary + detail views | `parsli.services.dashboard_projection_service` |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))

import json
import time
from datetime import datetime, timezone
from typing import Literal

import pandas as pd

from parsli.config import AppConfig, GmailConfig, ModelConfig
from parsli.gmail.auth import GmailOAuthManager, TokenMissingError
from parsli.gmail.client import GmailClient
from parsli.gmail.candidate_fetcher import GmailCandidateFetcher
from parsli.gmail.query_builder import GmailQueryBuilder
from parsli.gmail.models import DomainPreferences
from parsli.gmail.sender_trust import SenderTrustScorer, SenderTrustLevel
from parsli.privacy.sanitizer import extract_sender_domain, clip_text
from parsli.privacy.hashing import sha256_hex

DATA_FILE = Path("data/emails.json")
DATA_FILE.parent.mkdir(exist_ok=True)

config = AppConfig(
    app_dir=PROJECT_ROOT / ".parsli",
    database={"sqlite_path": PROJECT_ROOT / ".parsli" / "parsli.db"},
)

print("project root :", PROJECT_ROOT)
print("credentials  :", config.credentials_path, "→", "✓" if config.credentials_path.exists() else "✗ NOT FOUND")
print("tokens dir   :", config.tokens_dir)
print("db           :", config.database.sqlite_path)
print("model        :", config.model.provider, "/", config.model.model_name)

In [ ]:
# ── DB setup — run once, available across all parts ───────────────────────────
from parsli.db.session import ensure_schema, make_engine, make_session_factory

engine          = make_engine(config.database.sqlite_path)
ensure_schema(engine)   # create_all + ALTER TABLE for new observability columns
session_factory = make_session_factory(engine)

print(f"DB: {config.database.sqlite_path.resolve()}")

---
## Part 1 — Gmail auth + email download

In [ ]:
# List saved accounts so you can pick one
_oauth_list = GmailOAuthManager(
    credentials_path=config.credentials_path,
    tokens_dir=config.tokens_dir,
)
accounts = _oauth_list.list_token_accounts()

print("Saved accounts found:")
for a in accounts:
    print(f"  {a}")

ACCOUNT_ID = accounts[0] if accounts else ""
print(f"\nUsing:")

In [ ]:
oauth = GmailOAuthManager(
    credentials_path=config.credentials_path,
    tokens_dir=config.tokens_dir,
)

try:
    creds = oauth.refresh_if_needed(ACCOUNT_ID)
    print(f"Token loaded for: {ACCOUNT_ID}")
except TokenMissingError:
    print("No valid token — starting browser OAuth flow…")
    from google_auth_oauthlib.flow import InstalledAppFlow
    from googleapiclient.discovery import build as _gmail_build
    flow = InstalledAppFlow.from_client_secrets_file(
        str(config.credentials_path),
        scopes=GmailOAuthManager.SCOPES,
    )
    creds = flow.run_local_server(port=0)
    _profile = _gmail_build("gmail", "v1", credentials=creds).users().getProfile(userId="me").execute()
    ACCOUNT_ID = _profile["emailAddress"]
    oauth.save_token(ACCOUNT_ID, creds)
    print(f"Token saved for: {ACCOUNT_ID}")

In [ ]:
# ── Gmail query config — edit lookback_days or enabled languages here ─────────
from parsli.languages import load_language_packs
from parsli.services.domain_preference_service import DomainPreferenceService

gmail_cfg = GmailConfig(lookback_days=90)

# Language packs drive all query terms (shipping signals, order lifecycle,
# exclude terms, etc.). Add codes like "ru" or "pt" once those packs exist.
lang_config = load_language_packs(config.language.enabled)

with session_factory() as _s:
    domain_prefs = DomainPreferenceService(_s).get_preferences()

# Auto-add own email to exclude_senders if not already present
if ACCOUNT_ID and '@' in ACCOUNT_ID and ACCOUNT_ID not in domain_prefs.exclude_senders:
    with session_factory() as _s:
        DomainPreferenceService(_s).add_exclude_sender(ACCOUNT_ID)
        _s.commit()
    with session_factory() as _s:
        domain_prefs = DomainPreferenceService(_s).get_preferences()
    print(f"Auto-added {ACCOUNT_ID} to exclude_senders")

builder = GmailQueryBuilder(gmail_cfg, domain_prefs, lang_config=lang_config)
queries = builder.build_queries()

print(f"Active languages: {lang_config.active_codes}")
print(f"Allowlist       : {domain_prefs.allowlist or '(empty)'}")
print(f"Blocklist       : {domain_prefs.blocklist or '(empty)'}")
print(f"Exclude senders : {domain_prefs.exclude_senders or '(empty)'}")
print(f"\n{len(queries)} named queries:\n")
for q in queries:
    print(f"[{q.name}]")
    print(f"  {q.query}")
    print()

In [ ]:
# ── Manage domain/sender preferences ─────────────────────────────────────────
# Your own email is excluded automatically on the first sync — no action needed.
# Use this cell to add extra domains or senders, or to remove existing entries.
# Re-run the query config cell below after making any changes.

from parsli.services.domain_preference_service import DomainPreferenceService

# ── Edit these lists ──────────────────────────────────────────────────────────
ADD_EXCLUDE_SENDERS:    list[str] = []   # extra senders to exclude by email address
REMOVE_EXCLUDE_SENDERS: list[str] = []

ADD_BLOCKLIST:    list[str] = []         # whole domains to block from all queries
REMOVE_BLOCKLIST: list[str] = []

ADD_ALLOWLIST:    list[str] = []         # domains to get a dedicated allowlist query
REMOVE_ALLOWLIST: list[str] = []
# ─────────────────────────────────────────────────────────────────────────────

with session_factory() as _s:
    svc = DomainPreferenceService(_s)
    for s in ADD_EXCLUDE_SENDERS:    svc.add_exclude_sender(s)
    for s in REMOVE_EXCLUDE_SENDERS: svc.remove_exclude_sender(s)
    for d in ADD_BLOCKLIST:          svc.add_blocklist(d)
    for d in REMOVE_BLOCKLIST:       svc.remove_blocklist(d)
    for d in ADD_ALLOWLIST:          svc.add_allowlist(d)
    for d in REMOVE_ALLOWLIST:       svc.remove_allowlist(d)
    updated = svc.get_preferences()
    _s.commit()

print(f"Exclude senders : {updated.exclude_senders or '(set automatically on first sync)'}")
print(f"Blocklist       : {updated.blocklist or '(empty)'}")
print(f"Allowlist       : {updated.allowlist or '(empty)'}")

In [ ]:
client = GmailClient(creds)

fetcher    = GmailCandidateFetcher(client=client, builder=builder)
candidates = fetcher.fetch()

print(f"Unique candidates : {len(candidates.message_ids)}")
print(f"Matched >1 query  : {candidates.summary.multi_query_matches}")
print(f"Batch ID          : {candidates.summary.fetch_batch_id}")

# Keep a flat list for the download step below
message_ids = candidates.message_ids

In [ ]:
# ── Persist observability data ────────────────────────────────────────────────
# Saves query run timings and per-message source matches to the DB.
# Required for CandidateObservabilityService queries later in the session.
from parsli.services.candidate_observability_service import persist_fetch_result

with session_factory() as _s:
    persist_fetch_result(_s, candidates)
    _s.commit()

print("Query runs and candidate matches saved to DB.")

In [ ]:
# ── Query combination breakdown ───────────────────────────────────────────────
# candidates.summary is computed by the fetcher — no recomputation needed.
from collections import Counter

summary = candidates.summary

print("Per-query totals (use these with INSPECT_GROUP):")
print(f"  {'Query name':<32} {'Contributed':>11}  {'Exclusive':>9}")
print("  " + "─" * 57)

combo_counts = Counter(
    " + ".join(sorted(v)) for v in candidates.query_sources_by_message_id.values()
)
for name, total in sorted(summary.query_result_counts.items(), key=lambda x: -x[1]):
    exclusive = combo_counts.get(name, 0)
    print(f"  {name:<32} {total:>11}  {exclusive:>9}")

print()
print("Exact combination counts:")
print(f"  {'Combination':<50} {'Count':>6}")
print("  " + "─" * 58)
for combo, count in sorted(combo_counts.items(), key=lambda x: -x[1]):
    print(f"  {combo:<50} {count:>6}")
print("  " + "─" * 58)
print(f"  {'TOTAL':<50} {summary.total_unique_candidates:>6}")
print(f"  (batch: {summary.fetch_batch_id})")

In [ ]:
LIMIT = 300  # reduce for faster iteration

emails = []
for i, msg_id in enumerate(message_ids[:LIMIT]):
    raw = client.fetch_raw(msg_id)
    headers = client.extract_headers(raw)
    body = client.extract_body(raw.get('payload', {}))
    emails.append({
        'id':               msg_id,
        'subject':          headers.get('Subject', ''),
        'sender':           headers.get('From', ''),
        'date':             headers.get('Date', ''),
        'body':             body,
        'internal_date_ms': int(raw.get('internalDate', 0)),
    })
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{min(LIMIT, len(message_ids))}")

DATA_FILE.write_text(json.dumps(emails, ensure_ascii=False, indent=2))
print(f"\nSaved {len(emails)} emails → {DATA_FILE.resolve()}")

In [ ]:
# Inspect one email — change IDX to browse others
IDX = 0
e = emails[IDX]
print(f"Subject : {e['subject']}")
print(f"From    : {e['sender']}")
print(f"Date    : {e['date']}")
print(f"\nBody preview:\n{e['body'][:600]}")

In [ ]:
# ── Inspect emails by query group ────────────────────────────────────────────
# INSPECT_GROUP accepts:
#   - a single query name  →  "weak_phrases"   (all emails that matched it, including combos)
#   - an exact combo       →  "strong_shipping + weak_phrases"  (exact match only)
#
# SAVE_JSON = True  writes data/inspect_<group>.json for offline review.
# The JSON stores subject/sender/date/query_sources + a 300-char body preview
# (no full body stored to disk by default).

INSPECT_GROUP = "weak_phrases"   # ← edit this
SAVE_JSON     = True

# ── helpers ───────────────────────────────────────────────────────────────────
def _combo_key(sources: list[str]) -> str:
    return " + ".join(sorted(sources))

def _matches_group(sources: list[str], group: str) -> bool:
    if "+" in group:
        return _combo_key(sources) == group.strip()
    return group.strip() in sources

# ── filter ────────────────────────────────────────────────────────────────────
id_to_email  = {e["id"]: e for e in emails}
matched_ids  = [
    mid for mid, srcs in candidates.query_sources_by_message_id.items()
    if _matches_group(srcs, INSPECT_GROUP) and mid in id_to_email
]
matched      = [
    {**id_to_email[mid], "_sources": candidates.query_sources_by_message_id[mid]}
    for mid in matched_ids
]

print(f"Group '{INSPECT_GROUP}': {len(matched)} emails")

# ── save JSON ─────────────────────────────────────────────────────────────────
if SAVE_JSON and matched:
    slug     = INSPECT_GROUP.replace(" + ", "_").replace(" ", "_")
    out_path = Path(f"data/inspect_{slug}.json")
    payload  = [
        {
            "id":           e["id"],
            "subject":      e.get("subject", ""),
            "sender":       e.get("sender", ""),
            "date":         e.get("date", ""),
            "query_sources": e["_sources"],
            "body_preview": e.get("body", "")[:300],
        }
        for e in matched
    ]
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    print(f"Saved  → {out_path.resolve()}")

# ── display ───────────────────────────────────────────────────────────────────
pd.set_option("display.max_colwidth", 80)
pd.DataFrame([
    {
        "subject":      e.get("subject", "")[:70],
        "sender":       e.get("sender", "")[:45],
        "date":         e.get("date", "")[:16],
        "queries":      ", ".join(e["_sources"]),
        "body_preview": e.get("body", "")[:120].replace("\n", " "),
    }
    for e in matched
])

---
## Part 2 — Deterministic preprocessing

`EmailCleaner` strips HTML/boilerplate and returns a `CleanedEmail` DTO.  
`RuleEngine` applies regex + phrase rules and returns a `RuleExtractionResult` DTO.  
No model calls happen here.

In [ ]:
# Run this cell to reload from disk without re-downloading
emails = json.loads(DATA_FILE.read_text())
print(f"Loaded {len(emails)} emails from {DATA_FILE}")

In [ ]:
from parsli.languages import load_language_packs
from parsli.processing.cleaner import EmailCleaner
from parsli.processing.rule_engine import RuleEngine

# Share one merged config so cleaner + rule engine use identical patterns.
# lang_config may already exist from the query-config cell above; rebuild
# here so this cell runs independently when reloading from disk.
lang_config  = load_language_packs(config.language.enabled)
cleaner      = EmailCleaner(lang_config)
rule_engine  = RuleEngine(lang_config)
# Use the user's blocklist so blocked domains score as BLOCKED (not LOW)
_blocklist   = frozenset(getattr(globals().get('domain_prefs'), 'blocklist', []))
trust_scorer = SenderTrustScorer(user_blocklist=_blocklist)

preprocessed_rows = []
for e in emails:
    cleaned       = cleaner.clean(e['id'], e.get('body', ''))
    sender_domain = extract_sender_domain(e.get('sender', ''))
    rules         = rule_engine.extract(e['id'], cleaned.cleaned_text, sender_domain=sender_domain, subject=e.get('subject', ''))
    trust         = trust_scorer.score(sender_domain)
    preprocessed_rows.append({
        'email_id':             e['id'],
        'subject':              e.get('subject', ''),
        'sender':               e.get('sender', ''),
        'date':                 e.get('date', ''),
        'internal_date_ms':     e.get('internal_date_ms', 0),
        'sender_domain':        sender_domain,
        # sender trust
        'trust_level':          trust.trust_level.value,
        'trust_score':          trust.trust_score,
        'trust_reasons':        trust.reasons,
        # cleaner outputs
        'cleaned_text':         cleaned.cleaned_text,
        'cleaned_full_len':     cleaned.cleaned_full_len,
        'cleaned_text_hash':    cleaned.cleaned_text_hash,
        'is_shipping_shaped':   cleaned.is_shipping_shaped,
        # rule outputs
        'rule_status':          rules.status.value if rules.status else None,
        'rule_confidence':      rules.status_confidence,
        'rule_evidence':        rules.status_evidence,
        'is_invoice':           rules.is_invoice,
        'is_shipping_email':    rules.is_shipping_email,
        'tracking_candidates':  [t.value for t in rules.tracking_candidates],
        'order_candidates':     [o.value for o in rules.order_candidates],
        'selected_tracking_number': rules.tracking_candidates[0].value if rules.tracking_candidates else None,
        'selected_order_number':    rules.order_candidates[0].value if rules.order_candidates else None,
        'merchant':             rules.merchant,
        'pickup_code':          rules.pickup_code,
        'amount':               rules.amount,
        'currency':             rules.currency,
    })

pre_df = pd.DataFrame(preprocessed_rows)

print(f"Active languages: {lang_config.active_codes}")
print(f"Preprocessed:     {len(pre_df)}")
print(f"Shipping-shaped:  {pre_df['is_shipping_shaped'].sum()}")
print(f"Rule-classified:  {pre_df['rule_status'].notna().sum()}")
print(f"Invoices/payment: {pre_df['is_invoice'].sum()}")
print(f"\nSender trust distribution:")
print(pre_df['trust_level'].value_counts().to_string())

In [ ]:
pre_df[[
    'subject', 'sender_domain',
    'trust_level', 'trust_score',
    'is_shipping_shaped', 'is_invoice',
    'rule_status', 'rule_confidence',
    'tracking_candidates', 'order_candidates',
]].head(20)
pre_df.to_json('data/preprocessed.json')

---
## Part 3 — Model classification

Rows the rule engine can't decide (`rule_status` is `None` or `unknown`) are
sent to the local model. Backend is switchable: `lmstudio` or `llamacpp`.

In [ ]:
from parsli.model.factory import ModelClientFactory
from parsli.model.base import ModelClassificationResult, ModelAuditResult
from parsli.model.prompts import format_required_prompt, format_audit_prompt, build_model_text_preview
from parsli.processing.model_classifier import ModelClassifier, ModelExecutionMode, ModelCallObservability
from parsli.processing.reconciler import ClassificationReconciler, FinalClassificationResult, DecisionSource
from parsli.processing.cleaner import CleanedEmail
from parsli.processing.rule_engine import RuleExtractionResult
from parsli.domain.statuses import ShipmentStatus
from parsli.domain.identifiers import TrackingIdentifier, OrderIdentifier
from parsli.privacy.debug_store import DebugStore

# ── Edit these ────────────────────────────────────────────────────────────────
BACKEND: Literal['lmstudio', 'llamacpp'] = 'lmstudio'

model_cfg = ModelConfig(
    provider=BACKEND,
    model_name='google/gemma-4-e4b',    # must match the model loaded in LM Studio
    endpoint_url='http://127.0.0.1:1234',
    timeout_seconds=120,
    required_max_chars=4000,
    audit_max_chars=1500,
)
# ─────────────────────────────────────────────────────────────────────────────

# One persistent HTTP client for the whole session — reused across all calls.
model_client = ModelClientFactory.create(model_cfg)
print(f"Backend: {BACKEND}  model: {model_cfg.model_name}")
print(f"Text limits: required={model_cfg.required_max_chars}  audit={model_cfg.audit_max_chars}")

In [ ]:
# ── Smoke test — run this cell alone to verify the model is reachable ─────────
# This cell is intentionally separate from the batch loop below.
# Do NOT include it in the main batch path.

from parsli.model.prompts import format_required_prompt, build_model_text_preview
from parsli.model.base import ModelClassificationResult

_test  = next((r for r in preprocessed_rows if r['is_shipping_shaped']), preprocessed_rows[0])
_prev  = build_model_text_preview(_test['cleaned_text'], max_chars=4000)
_prompt = format_required_prompt(
    subject=_test.get('subject', ''),
    sender_domain=_test.get('sender_domain'),
    email_text=_prev,
)
_t0     = time.perf_counter()
_res    = model_client.extract(_prompt, response_model=ModelClassificationResult)
_elapsed = time.perf_counter() - _t0

print(f"email_type={_res.email_type.value}  status={_res.status.value}  confidence={_res.status_confidence:.2f}  ({_elapsed:.1f}s)")
print(f"evidence  : {_res.status_evidence[:120]}")
print(f"carrier   : {_res.carrier}   merchant: {_res.merchant}")
print(f"reasoning : {_res.reasoning}")
_usage = getattr(model_client, 'last_usage', None) or {}
print(f"tokens    : prompt={_usage.get('prompt_tokens')}  completion={_usage.get('completion_tokens')}")
print(f"prompt_len: {len(_prompt)} chars  preview_len: {len(_prev)} chars")

In [ ]:
# ── Classification loop: rules → ModelClassifier → ClassificationReconciler ───
# Exactly one model call per email. The reconciler reuses the returned
# model_result — it does not call the model again.

_debug      = DebugStore(app_dir=config.app_dir, enabled=False)
_classifier = ModelClassifier(
    model_client=model_client,
    model_provider=BACKEND,
    model_name=model_cfg.model_name,
    model_config=model_cfg,
    debug_store=_debug,
)
_reconciler = ClassificationReconciler()

results: list[FinalClassificationResult] = []

# Per-call observability log
call_log: list[dict] = []
rule_only = model_audit = model_required = errors = 0

for i, pre in enumerate(preprocessed_rows, 1):
    _cleaned = CleanedEmail(
        email_id=pre['email_id'],
        cleaned_text=pre['cleaned_text'],
        cleaned_text_hash=pre['cleaned_text_hash'],
        cleaned_full_len=pre['cleaned_full_len'],
        is_shipping_shaped=pre['is_shipping_shaped'],
        subject=pre.get('subject', ''),
        sender_domain=pre.get('sender_domain'),
    )
    _rules = RuleExtractionResult(
        is_shipping_email=pre['is_shipping_email'],
        is_invoice=pre['is_invoice'],
        status=ShipmentStatus(pre['rule_status']) if pre['rule_status'] else None,
        status_confidence=pre['rule_confidence'],
        status_evidence=pre['rule_evidence'],
        tracking_candidates=[TrackingIdentifier(value=v) for v in (pre['tracking_candidates'] or [])],
        order_candidates=[OrderIdentifier(value=v) for v in (pre['order_candidates'] or [])],
        merchant=pre['merchant'],
        pickup_code=pre['pickup_code'],
        amount=pre['amount'],
        currency=pre['currency'],
    )

    mode = _classifier.select_mode(_rules, _cleaned, sender_trust_level=pre['trust_level'])

    try:
        # Single model call per email — rules passed for audit mode context
        model_result, obs = _classifier.classify(_cleaned, mode, rules=_rules)
    except Exception as exc:
        print(f"  [{i}] classifier error: {exc}")
        model_result = None
        obs = ModelCallObservability(mode=mode, called=True, prompt_type=None)
        errors += 1

    final = _reconciler.reconcile(
        rules=_rules,
        model=model_result,
        obs=obs,
        cleaned=_cleaned,
        email_id=pre['email_id'],
        processing_version=config.processing.processing_version(),
        model_provider=BACKEND,
        model_name=model_cfg.model_name,
        privacy=config.privacy,
    )
    results.append(final)

    call_log.append({
        "email_id":   pre['email_id'],
        "mode":       mode.value,
        "called":     obs.called,
        "prompt_type": obs.prompt_type,
        "latency_ms": obs.latency_ms,
    })

    if mode == ModelExecutionMode.SKIP_MODEL:
        rule_only += 1
    elif mode == ModelExecutionMode.MODEL_AUDIT:
        model_audit += 1
    else:
        model_required += 1

# ── Observability summary ─────────────────────────────────────────────────────
model_calls = sum(1 for e in call_log if e["called"])
emails_processed = len(preprocessed_rows)

print(f"emails_processed={emails_processed}")
print(f"model_calls={model_calls}  required={model_required}  audit={model_audit}  skip={rule_only}  errors={errors}")

assert model_calls == model_required + model_audit, (
    f"model_calls ({model_calls}) ≠ required ({model_required}) + audit ({model_audit})"
)
assert model_calls <= emails_processed, (
    f"model_calls ({model_calls}) > emails_processed ({emails_processed})"
)
print("✓ one model call per email assertion passed")

In [ ]:
RESULTS_CSV = Path('data/results.json')

# Join FinalClassificationResult objects with per-email metadata from Part 2
_pre_by_id = {r['email_id']: r for r in preprocessed_rows}

rows_for_df = []
for final in results:
    pre = _pre_by_id.get(final.email_id, {})
    rows_for_df.append({
        'email_id':               final.email_id,
        'subject':                pre.get('subject', ''),
        'sender_domain':          pre.get('sender_domain', ''),
        'trust_level':            pre.get('trust_level', ''),
        # Email classification
        'email_type':             final.email_type.value,
        'rule_email_type':        final.rule_email_type.value,
        'model_email_type':       final.model_email_type.value if final.model_email_type else None,
        # Shipment status
        'status':                 final.status.value,
        'rule_status':            final.rule_status.value if final.rule_status else None,
        'model_status':           final.model_status.value if final.model_status else None,
        'status_confidence':      final.status_confidence,
        'status_evidence':        final.status_evidence[:120],
        # Per-source confidence
        'rule_confidence':        final.rule_confidence,
        'model_confidence':       final.model_confidence,
        # Relevance
        'is_relevant':            final.is_relevant,
        'is_invoice':             final.is_invoice,
        'ignore_reason':          final.ignore_reason,
        # Decision metadata
        'decision_source':        final.decision_source.value,
        'conflict_reason':        final.conflict_reason,
        'needs_review':           final.needs_review,
        'classification_method':  final.classification_method,
        # Extraction fields
        'carrier':                final.carrier,
        'merchant':               final.merchant,
        'selected_tracking_number': final.selected_tracking_number,
        'tracking_candidates':    '|'.join(t.value for t in final.tracking_candidates),
        'selected_order_number':  final.selected_order_number,
        'pickup_code':            final.pickup_code,
        'amount':                 final.amount,
        'currency':               final.currency,
        # Observability
        'model_called':           final.model_called,
        'model_mode':             final.model_mode,
        'model_latency_ms':       round(final.model_latency_ms) if final.model_latency_ms else None,
        'prompt_tokens':          final.prompt_tokens,
        'completion_tokens':      final.completion_tokens,
        'rule_model_agreed':      final.rule_model_agreed,
        'confidence_delta':       round(final.confidence_delta, 3) if final.confidence_delta else None,
    })

df = pd.DataFrame(rows_for_df)
df.to_json(RESULTS_CSV, index=False)
print(f"Saved → {RESULTS_CSV.resolve()}")

pd.set_option('display.max_colwidth', 80)
df[[
    'subject', 'sender_domain',
    'email_type', 'status', 'status_confidence',
    'decision_source', 'conflict_reason',
    'selected_tracking_number', 'tracking_candidates',
]].head(30)

In [ ]:
print("=== Email type distribution ===")
print(df['email_type'].value_counts().to_string())

print("\n=== Status distribution ===")
print(df['status'].value_counts().to_string())

print("\n=== Decision source ===")
print(df['decision_source'].value_counts().to_string())

print("\n=== Model mode ===")
print(df['model_mode'].value_counts().to_string())

print("\n=== Classification method ===")
print(df['classification_method'].value_counts().to_string())

# Conflicts: model_fallback, semantic_guard, rule_override, model_override
conflict_sources = ['model_fallback', 'rule_override', 'model_override', 'semantic_guard', 'review_needed']
conflicts = df[df['decision_source'].isin(conflict_sources)]
print(f"\nNon-agreement decisions ({len(conflicts)}):")
if len(conflicts):
    print(conflicts[[
        'subject', 'decision_source',
        'rule_email_type', 'model_email_type',
        'rule_confidence', 'model_confidence',
        'conflict_reason',
    ]].to_string())

# Needs review
needs_review_df = df[df['needs_review'] == True]
print(f"\nNeeds review: {len(needs_review_df)}")
if len(needs_review_df):
    print(needs_review_df[[
        'subject', 'decision_source', 'rule_email_type', 'model_email_type',
        'rule_confidence', 'model_confidence', 'conflict_reason',
    ]].to_string())

model_called_df = df[df['model_called'] == True]
print(f"\nModel called: {len(model_called_df)}  avg latency: {model_called_df['model_latency_ms'].mean():.0f} ms")
if model_called_df['prompt_tokens'].notna().any():
    print(f"Avg tokens  : prompt={model_called_df['prompt_tokens'].mean():.0f}  completion={model_called_df['completion_tokens'].mean():.0f}")

digital = df[df['email_type'] == 'digital_product']
print(f"\nDigital product emails: {len(digital)}")
if len(digital):
    print(digital[['subject', 'sender_domain']].to_string())

---
## Part 4 — Persistence + shipment resolution

`ShipmentResolutionService` resolves canonical shipment IDs, inserts
`shipment_events` rows (idempotent), and rebuilds the `shipments` timeline.

Routing by email type:

| email_type | Identifier required | What gets created |
|---|---|---|
| `order_confirmation` | `order_number` (+ `merchant` when available) | order alias + `order_confirmed` event (no tracking alias invented) |
| `shipping_update` / `pickup_ready` / `delivered` / `payment_problem` | `tracking_number` or `order_number` | tracking alias (merge-checked), order alias, shipment event |
| `digital_product` / `billing_only` / `non_shipping` | — | nothing (skipped) |

If a `shipping_update` arrives for the same `(order_number, merchant)` that an
`order_confirmation` already created, the tracking alias is attached to the
**existing** canonical — no second shipment is created.

In [ ]:
from parsli.db.repositories import EmailAccountRepository
from parsli.services.shipment_resolution_service import ShipmentResolutionService
from parsli.privacy.sanitizer import extract_sender_display_name, extract_sender_domain

# engine / session_factory already created in the DB setup cell at the top.
print(f"DB: {config.database.sqlite_path.resolve()}")

_received_at = {
    e['id']: datetime.fromtimestamp(e['internal_date_ms'] / 1000, tz=timezone.utc)
    for e in emails
}

# Build sender info lookup from the raw 'From' header stored in the emails list.
# sender_display_name: e.g. "Care to Beauty" from "Care to Beauty <help@caretobeauty.com>"
# sender_domain:       e.g. "caretobeauty.com"
_sender_info = {
    e['id']: (
        extract_sender_display_name(e.get('sender', '')),
        extract_sender_domain(e.get('sender', '')),
    )
    for e in emails
}

EMAIL_ACCOUNT_HASH = sha256_hex(ACCOUNT_ID)

with session_factory() as session:
    acct_repo = EmailAccountRepository(session)
    account   = acct_repo.find_by_hash(EMAIL_ACCOUNT_HASH)
    if account is None:
        account = acct_repo.create(EMAIL_ACCOUNT_HASH)

    resolution_svc = ShipmentResolutionService(session, config.processing)
    inserted = skipped = 0

    # `results` is the list[FinalClassificationResult] produced by Part 3.
    for final in results:
        display_name, domain = _sender_info.get(final.email_id, (None, None))
        resolution_svc.resolve_and_insert(
            final,
            _received_at.get(final.email_id, datetime.now(timezone.utc)),
            sender_display_name=display_name,
            sender_domain=domain,
        )
        if final.is_relevant:
            inserted += 1
        else:
            skipped += 1

    session.commit()
    print(f"Committed — {inserted} events inserted, {skipped} skipped (irrelevant/invoice)")

In [ ]:
from parsli.services.dashboard_service import DashboardService

with session_factory() as session:
    dash = DashboardService(session).get_dashboard()

print(f"Total: {dash.total_count}  Active: {dash.active_count}  Delivered: {dash.delivered_count}\n")

res = pd.DataFrame([{
    'id':           s.canonical_shipment_id,
    'merchant':     s.merchant,
    'tracking':     s.primary_tracking_number,
    'order':        s.primary_order_number,
    'status':       s.current_status.value,
    'status_date':  s.current_status_date.date(),
    'evidence':     s.current_status_evidence[:80],
    'events':       s.event_count,
    'chronology':   s.chronology_severity,
} for s in dash.shipments])

res.to_json("data/afterresolution.json")

In [ ]:
# Chronology issues
issues = [s for s in dash.shipments if s.chronology_severity != 'ok']
print(f"Chronology issues: {len(issues)}")
for s in issues:
    print(f"\n  [{s.chronology_severity.upper()}] {s.canonical_shipment_id}")
    print(f"  tracking={s.primary_tracking_number}  merchant={s.merchant}")
    for note in s.chronology_notes:
        print(f"    • {note}")

---
## Part 5 — Dashboard projection

`DashboardProjectionService` reads from the already-resolved tables and builds
UI-ready projection objects. No business logic runs here — pure read + project.

| Kind | Meaning |
|---|---|
| `tracked` | Shipment has a carrier tracking number |
| `order_only` | Order confirmation received, no carrier tracking yet |

`needs_review` is `True` when either the chronology has a warning/conflict
**or** any of the email extractions for this shipment flagged `needs_review`.

API endpoints added:
- `GET /api/dashboard/projection` → `DashboardProjection`
- `GET /api/shipments/{id}/detail` → `ShipmentDetailProjection`

In [ ]:
# ── Dashboard projection — summary list + counts ──────────────────────────────
from parsli.services.dashboard_projection_service import DashboardProjectionService

with session_factory() as session:
    proj_svc = DashboardProjectionService(session)
    proj = proj_svc.get_dashboard_projection()

print(f"Total: {proj.total_count}  Active: {proj.active_count}  Delivered: {proj.delivered_count}")
print(f"Order-only: {proj.order_only_count}  Needs review: {proj.needs_review_count}\n")

proj_df = pd.DataFrame([{
    'kind':         r.shipment_kind,
    'title':        r.display_title,
    'merchant':     r.merchant,
    'status':       r.current_status_label,
    'chronology':   r.chronology_status,
    'reason':       r.chronology_reason,
    'date':         r.last_status_date.date(),
    'events':       r.events_count,
    'needs_review': r.needs_review,
} for r in proj.shipments])


proj_df.to_json("data/projection.json")
proj_df

In [ ]:
# ── Shipment detail — full timeline for one shipment ──────────────────────────
# Change INSPECT_IDX to browse other shipments.
INSPECT_IDX = 0

if proj.shipments:
    target = proj.shipments[INSPECT_IDX]
    with session_factory() as session:
        detail = DashboardProjectionService(session).get_shipment_detail(target.shipment_id)

    if detail:
        flag = " ⚠ NEEDS REVIEW" if detail.needs_review else ""
        print(f"[{detail.shipment_kind.upper()}] {detail.display_title}{flag}")
        print(f"  Status     : {detail.current_status_label}  ({detail.chronology_status})")
        print(f"  Events     : {len(detail.events)}  |  merge_confidence={detail.merge_confidence:.2f}")
        print(f"  First seen : {detail.first_seen_at.date()}  →  last: {detail.last_status_date.date()}\n")

        print(f"  {'Date':<12} {'Status':<32} {'Source':<18} {'Evidence'}")
        print("  " + "─" * 100)
        for ev in detail.events:
            src = ev.decision_source or "—"
            evidence = ev.status_evidence[:55] if ev.status_evidence else "—"
            review_flag = " ⚠" if ev.needs_review else ""
            print(f"  {str(ev.event_date.date()):<12} {ev.status_label:<32} {src:<18} {evidence}{review_flag}")
else:
    print("No shipments resolved yet — run Part 4 first.")